# MP3: Modelowanie bazowe i porównanie algorytmów

**MajsterPlus — Predykcja rezygnacji klientów**

W tym mini-projekcie:
1. Wytrenujesz bazowy model Logistic Regression
2. Wytrenujesz klasyfikator Random Forest
3. Ocenisz oba modele (confusion matrix, classification report, ROC-AUC)
4. Porównasz modele i przeanalizujesz feature importance
5. Zinterpretujesz wyniki w kontekście biznesowym MajsterPlus

**Faza CRISP-DM**: Modelowanie

---

## 0. Konfiguracja i Reprodukowalność

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Asercje wersji bibliotek
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — oczekiwano 1.4.x lub 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — oczekiwano 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — oczekiwano <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Wczytanie danych (checkpoint z MP2)

In [ ]:
import hashlib, pickle
from pathlib import Path

# Wykrywanie środowiska Colab i montowanie Google Drive
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/PUM/checkpoints")
except ImportError:
    DATA_DIR = Path("../2. data")
    CHECKPOINT_DIR = Path("../checkpoints")

checkpoint_file = CHECKPOINT_DIR / "mp2_checkpoint.pkl"

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"Wczytano checkpoint z MP2: {list(checkpoint.keys())}")
else:
    raise FileNotFoundError(
        f"Nie znaleziono checkpoint pod ścieżką {checkpoint_file}\n\n"
        "Upewnij się, że na Google Drive istnieje odpowiednia struktura folderów:\n\n"
        "  Mój dysk/\n"
        "    PUM/\n"
        "      2. data/\n"
        "        customers.csv\n"
        "        transactions.csv\n"
        "      checkpoints/\n"
        "        mp2_checkpoint.pkl\n\n"
        "Jeśli nie masz checkpoint — najpierw uruchom notebook MP2.\n"
        "Jeśli folder 'PUM' nie istnieje — utwórz go na Google Drive\n"
        "i wgraj pliki z repozytorium kursu."
    )

In [ ]:
if checkpoint is not None:
    X_train = checkpoint["X_train"]
    X_test = checkpoint["X_test"]
    y_train = checkpoint["y_train"]
    y_test = checkpoint["y_test"]
    feature_names = checkpoint["feature_names"]
    gender_test = checkpoint.get("gender_test")
else:
    raise FileNotFoundError(
        "Nie znaleziono checkpoint. Uruchom najpierw rozwiązanie MP2 "
        "lub umieść mp2_checkpoint.pkl w katalogu checkpoints."
    )

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Wskaźnik rezygnacji y_train: {y_train.mean():.3f}")
print(f"Wskaźnik rezygnacji y_test:  {y_test.mean():.3f}")
print(f"Liczba cech: {X_train.shape[1]}")

## 2. Logistic Regression

**Zadanie**: Wytrenuj model Logistic Regression i oceń jego wyniki.

Kroki:
1. Wytrenuj LogisticRegression. Użyj `random_state=42` i `max_iter=1000` dla Reprodukowalności.
2. Wygeneruj predykcje (etykiety klas) i prawdopodobieństwa dla klasy pozytywnej na zbiorze testowym.
3. Wyświetl confusion matrix, classification report (z `digits=4`) oraz ROC-AUC score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score)

# TWÓJ KOD TUTAJ: Wytrenuj LogisticRegression. Użyj random_state=42 i max_iter=1000 dla reprodukowalności.
# TWÓJ KOD TUTAJ: Wygeneruj predykcje klas (y_pred_lr) i estymaty prawdopodobieństw (y_prob_lr) na zbiorze testowym.
#       Dla prawdopodobieństw użyj kolumny klasy pozytywnej (indeks 1).


# TWÓJ KOD TUTAJ: Wyświetl confusion matrix, classification report (digits=4) oraz ROC-AUC score.


### Confusion matrix — interpretacja biznesowa

**Zastanów się**: Przyjrzyj się wartościom w confusion matrix. W kontekście MajsterPlus:
- **TP** = klienci, którzy zrezygnowali, poprawnie zidentyfikowani → zespół marketingu MOŻE do nich dotrzeć
- **FN** = klienci, którzy zrezygnowali, POMINIĘCI → utracona szansa na reaktywację
- **FP** = aktywni klienci błędnie oznaczeni → zmarnowany budżet kampanii
- **TN** = aktywni klienci poprawnie zidentyfikowani → brak wymaganej akcji

**TWÓJ KOD TUTAJ**: Oblicz recall dla klasy 1 (rezygnacja). Jaki odsetek zagrożonych klientów zespół marketingu faktycznie by dotarł? Czy jest to akceptowalne?

In [ ]:
# TWÓJ KOD TUTAJ: Wyodrębnij TP i FN z confusion matrix.
# TWÓJ KOD TUTAJ: Oblicz recall = TP / (TP + FN).
# TWÓJ KOD TUTAJ: Wyświetl wartość recall i napisz krótką interpretację:
#       Jaki procent zagrożonych klientów dotarłby zespół marketingu?


## 3. Random Forest

**Zadanie**: Wytrenuj klasyfikator Random Forest i oceń go za pomocą tych samych metryk.

Kroki:
1. Wytrenuj RandomForestClassifier. Użyj `random_state=42` i `n_estimators=100`.
2. Wygeneruj predykcje i prawdopodobieństwa na zbiorze testowym.
3. Wyświetl confusion matrix, classification report (z `digits=4`) oraz ROC-AUC score.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TWÓJ KOD TUTAJ: Wytrenuj RandomForestClassifier. Użyj random_state=42 i n_estimators=100.
# TWÓJ KOD TUTAJ: Wygeneruj predykcje klas (y_pred_rf) i estymaty prawdopodobieństw (y_prob_rf) na zbiorze testowym.


# TWÓJ KOD TUTAJ: Wyświetl confusion matrix, classification report (digits=4) oraz ROC-AUC score.


## 4. Porównanie krzywych ROC

**Zadanie**: Narysuj krzywe ROC dla obu modeli na jednym wykresie. Umieść wartość AUC w legendzie.

Użyj `RocCurveDisplay.from_predictions()` dla każdego modelu i narysuj je na tych samych osiach. Dodaj przekątną linię przerywaną reprezentującą losowy klasyfikator (AUC = 0.5).

In [ ]:
from sklearn.metrics import RocCurveDisplay

# TWÓJ KOD TUTAJ: Utwórz figurę z jedną osią (axes).
# TWÓJ KOD TUTAJ: Użyj RocCurveDisplay.from_predictions() aby narysować krzywą ROC każdego modelu na tych samych osiach.
# TWÓJ KOD TUTAJ: Dodaj linię bazową losowego klasyfikatora (przekątna).
# TWÓJ KOD TUTAJ: Ustaw tytuł, legendę i wyświetl wykres.


### Porównanie z baseline

W MP1 bazowy model LogisticRegression osiągnął ROC-AUC ≈ 0.83 przy zaledwie 6 surowych cechach numerycznych i bez czyszczenia danych. Twój model LogisticRegression z MP3 używa pełnego zestawu cech po czyszczeniu, kodowaniu i skalowaniu z MP2.

**TWÓJ KOD TUTAJ**: Porównaj ROC-AUC modelu LR z MP3 z baseline z MP1 (~0.83). Co wielkość poprawy (lub jej brak) mówi o tym, skąd pochodzi sygnał predykcyjny?

In [ ]:
# TWÓJ KOD TUTAJ: Wyświetl AUC baseline z MP1 (~0.83) i AUC LR z MP3 obok siebie.
# TWÓJ KOD TUTAJ: Oblicz różnicę i napisz krótką interpretację:
#       Skąd pochodzi większość sygnału predykcyjnego?


## 5. Feature importance (Random Forest)

**Zadanie**: Wyodrębnij i zwizualizuj 15 najważniejszych cech z modelu Random Forest.

Użyj atrybutu `.feature_importances_` modelu, przypisz wartości ważności do nazw cech i utwórz poziomy wykres słupkowy dla 15 najważniejszych cech.

In [ ]:
# TWÓJ KOD TUTAJ: Pobierz nazwy cech z X_train (kolumny jeśli DataFrame, w przeciwnym razie użyj feature_names).
# TWÓJ KOD TUTAJ: Utwórz pandas Series z wartościami ważności indeksowanymi nazwami cech.
# TWÓJ KOD TUTAJ: Wybierz 15 najważniejszych cech.
# TWÓJ KOD TUTAJ: Utwórz poziomy wykres słupkowy (posortowany rosnąco dla czytelności).
# TWÓJ KOD TUTAJ: Wyświetl 15 najważniejszych cech i ich wartości ważności.


## 6. Sprawdzenie overfitting

**Zadanie**: Porównaj dokładność na zbiorze treningowym i testowym dla obu modeli. Czy któryś z modeli wykazuje overfitting?

Użyj metody `.score()` każdego modelu na danych treningowych i testowych.

In [ ]:
# TWÓJ KOD TUTAJ: Wyświetl dokładność na zbiorze treningowym i testowym dla Logistic Regression.
# TWÓJ KOD TUTAJ: Wyświetl dokładność na zbiorze treningowym i testowym dla Random Forest.
# TWÓJ KOD TUTAJ: Skomentuj, który model (jeśli którykolwiek) wykazuje oznaki overfitting.


### Interpretacja overfitting

**TWÓJ KOD TUTAJ**: Jeśli Random Forest osiąga ~100% dokładności na zbiorze treningowym, ale ~81% na zbiorze testowym, co ta różnica oznacza? Czy jest to oczekiwane zachowanie dla Random Forest? Czy ufałbyś temu modelowi bardziej czy mniej niż Logistic Regression, który osiąga ~82% zarówno na zbiorze treningowym, jak i testowym?

In [ ]:
# TWÓJ KOD TUTAJ: Napisz swoją interpretację jako komentarze lub instrukcję print.
# Rozważ:
#   - Dlaczego RF zapamiętuje dane treningowe?
#   - Czy mała różnica train/test sprawia, że LR jest bardziej godny zaufania?
#   - Co oznacza „generalizacja" w tym kontekście?


## 7. Podsumowanie porównania modeli

**Zadanie**: Utwórz tabelę porównawczą kluczowych metryk dla obu modeli.

Uwzględnij: Accuracy, Precision (klasa 1), Recall (klasa 1), F1-Score (klasa 1) oraz ROC-AUC.

In [ ]:
# TWÓJ KOD TUTAJ: Zbuduj DataFrame z metrykami dla obu modeli.
# TWÓJ KOD TUTAJ: Zaokrąglij wartości do 4 miejsc po przecinku.
# TWÓJ KOD TUTAJ: Wyświetl tabelę porównawczą.


### Perspektywa biznesowa

**TWÓJ KOD TUTAJ**: Na podstawie powyższej tabeli porównawczej, który model wstępnie poleciłbyś dla kampanii reaktywacyjnej MajsterPlus? Rozważ accuracy, recall i ROC-AUC.

*Wskazówka: W MP4 dowiemy się, że „najlepszy" model statystyczny nie zawsze jest najbardziej opłacalny. Miej to na uwadze.*

In [ ]:
# TWÓJ KOD TUTAJ: Napisz swoją wstępną rekomendację jako komentarze lub instrukcję print.
# Który model wypada lepiej pod względem metryk? Dlaczego?
# Co może się zmienić, gdy uwzględnimy koszty kampanii w MP4?


## 8. Zapis checkpoint dla MP4

In [ ]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_mp3 = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": list(X_train.columns) if hasattr(X_train, "columns") else feature_names,
    "gender_test": gender_test,
    # Odkomentuj po wytrenowaniu modeli:
    # "lr_model": lr,
    # "rf_model": rf,
    # "y_pred_lr": y_pred_lr,
    # "y_prob_lr": y_prob_lr,
    # "y_pred_rf": y_pred_rf,
    # "y_prob_rf": y_prob_rf,
}

with open(CHECKPOINT_DIR / "mp3_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint_mp3, f)
print(f"Checkpoint zapisany: {CHECKPOINT_DIR / 'mp3_checkpoint.pkl'}")

---
*Koniec MP3 — Przejdź do MP4: Ewaluacja modelu i wpływ biznesowy*